## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# T-shirt colours, week 1: planned stock as counts

**Goodness-of-Fit Test** · a worked example from the StatNex teaching guide

**The story.** A clothing brand ordered 4,200 T-shirts in nine colours for the season, following its stocking plan. After the first week's 120 sales, the buyer checks whether customers buy colours in the same proportions as the plan. The buyer hopes the test will confirm the plan, so that the CEO can be told that no reorder is needed.

**The question.** Do sales follow the stocking plan, or did we buy the wrong colours?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `04_crosstab/goodness_of_fit.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Colour,Units sold,Proportion,Planned stock,Planned proportion
Black,32.0,0.2667,800.0,0.1905
White,20.0,0.1667,700.0,0.1667
Navy,15.0,0.125,500.0,0.119
Grey,14.0,0.1167,500.0,0.119
Red,11.0,0.0917,400.0,0.0952
Green,8.0,0.0667,400.0,0.0952
Pastel pink,8.0,0.0667,400.0,0.0952
Mustard,5.0,0.0417,300.0,0.0714
Tie-dye,7.0,0.0583,200.0,0.0476
Total,120.0,1.0,4200.0,1.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per class: the observed count, and the expected count or proportion.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
names = data["Colour"].astype(str)
# A total row (Total, or its Japanese equivalent) is a margin, not a class: counting it would double every count,
# so it is left out. Spaces, underscores, case and full-width letters are ignored when
# names are compared (totals.py).
# is_total: defined in the helper cell totals above (totals.py in the code download)

keep = [not is_total(name) for name in names]
data = data[keep]
names = list(names[keep])
observed = data["Units sold"].to_numpy(float)
expected_given = data["Planned stock"].to_numpy(float)

## Goodness-of-fit test

In [ ]:
# The expected column may hold counts or proportions: either way it is rescaled to the
# observed total, because the hypothesis is about the shape, not the size.
# How to read it: each class adds (observed - expected)^2 / expected to the chi-square,
# so the contribution column shows where the fit fails. P is the chance of a total at
# least this large if the stated proportions were true. The test assumes independent
# observations and expected counts of about 5 or more in every class.
expected = expected_given / expected_given.sum() * observed.sum()
test = stats.chisquare(observed, f_exp=expected)
df = len(observed) - 1
table = pd.DataFrame({"observed": observed, "expected": expected,
                      "contribution": (observed - expected) ** 2 / expected}, index=names)
print(table)
print(f"\nchi-square = {test.statistic:.4f}, df = {df}, P = {test.pvalue:.4g}")

## Observed and expected counts

In [ ]:
Path("figures").mkdir(exist_ok=True)
figure, axis = plt.subplots(figsize=(7, 4))
positions = np.arange(1, len(names) + 1)
axis.plot(positions, observed, "o", label="observed")
axis.plot(positions, expected, "D", label="expected")
axis.set_xticks(positions)
axis.legend()
figure.tight_layout()
figure.savefig("figures/fit.png", dpi=120)

## What the result says

The goodness-of-fit test compares the 120 sales with what the stocking plan predicts. Here the plan is given as stock counts (4,200 shirts), which the test scales down to 120 sales.

Chi-square is 7.55 with 8 degrees of freedom, and the P value is 0.479. A P value is the chance of gaps at least this large if customers really bought in the plan's proportions. Almost half the time, luck alone gives gaps like these. So the sales do not contradict the plan.

That is not the same as confirming it. With only 120 sales, the test can easily miss a wrong plan. Black already has 26.7% of sales against a planned 19%. The buyer cannot yet tell the CEO that no reorder is needed.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.